# 06 — ARC expression across subtypes and cell types

**Input**: `05_annotated.h5ad`  
**Output**: `results/arc_*.csv` tables, figures in `figures/`

**Biological questions**:
1. Which subtype (ER+, HER2+, TNBC) has the highest / lowest ARC expression?
2. Within each subtype, which cell type expresses ARC most strongly?

### Honest expectations

ARC is canonically a neuronal immediate-early gene. Its expression in breast tumors is expected to be **sparse** — most cells will have zero counts. That's not a bug; that's biology. The signal, if it exists, will show up as:

- A small fraction of cells (likely <5% in most cell types) with non-zero ARC
- Potentially enriched in specific populations (e.g., stromal/fibroblast cells, or rarely in tumor cells under stress)

### Statistical approach

- **Descriptive per-cell statistics**: useful to see where ARC is expressed at all.
- **Pseudo-bulk analysis**: aggregate counts per (patient × cell type), then compare across subtypes. This respects the patient-level structure of the data (Squair et al. 2021, *Nature Communications* — per-cell DE tests dramatically inflate false positives).

We use the authors' `celltype_major` labels as the primary grouping (they're validated across the paper) and our `celltype_predicted` as a cross-check.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import arc_analysis as aa

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 4))
sns.set_style('white')

adata = sc.read_h5ad(cfg.H5AD_ANNOTATED)
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')
print(f'ARC present: {"ARC" in adata.var_names}')

In [ ]:
# Sanity checks before analysis
# Ensure subtype and cell type columns are present and categorical
subtype_col = cfg.SUBTYPE_COLUMN
celltype_col = cfg.CELLTYPE_COLUMN_AUTHORS  # primary

print(f'Subtype column: {subtype_col}')
print(adata.obs[subtype_col].value_counts())
print(f'\nCell type column: {celltype_col}')
print(adata.obs[celltype_col].value_counts())

## 1. Descriptive: global ARC expression summary

In [ ]:
# Per-subtype summary (pool all cell types within a subtype)
summary_subtype = aa.summarize_gene_by_group(
    adata,
    gene='ARC',
    group_cols=(subtype_col,),
)
print('=== ARC expression by subtype (pooled across all cell types) ===')
display(summary_subtype)
summary_subtype.to_csv(cfg.RESULTS_DIR / 'arc_summary_by_subtype.csv', index=False)

These results are consistent with previous paper: “ER-positive/HER2-negative and luminal A expressed significantly higher ARC compared to the other subtypes” (Yee at al., 2025)

In [ ]:
# Per-cell-type summary (pool all subtypes)
summary_celltype = aa.summarize_gene_by_group(
    adata,
    gene='ARC',
    group_cols=(celltype_col,),
)
summary_celltype = summary_celltype.sort_values('mean_expr', ascending=False)
print('=== ARC expression by cell type (pooled across subtypes) ===')
display(summary_celltype)
summary_celltype.to_csv(cfg.RESULTS_DIR / 'arc_summary_by_celltype.csv', index=False)

In [ ]:
# Per-subtype x cell-type summary (primary table for the biological question)
summary_st_ct = aa.summarize_gene_by_group(
    adata,
    gene='ARC',
    group_cols=(subtype_col, celltype_col),
)
print('=== ARC expression by subtype x cell type ===')
display(summary_st_ct)
summary_st_ct.to_csv(cfg.RESULTS_DIR / 'arc_summary_by_subtype_celltype.csv', index=False)

## 2. Top ARC-expressing cell types per subtype

In [ ]:
top_by_mean = aa.top_cell_types_per_subtype(
    summary_st_ct,
    subtype_col=subtype_col,
    celltype_col=celltype_col,
    rank_by='mean_expr',
    top_n=3,
    min_cells=20,
)
print('=== Top-3 ARC-expressing cell types per subtype (by mean expression) ===')
display(top_by_mean)
top_by_mean.to_csv(cfg.RESULTS_DIR / 'arc_top_celltypes_by_mean.csv', index=False)

In [ ]:
# Ranking by % expressing (a zero-inflation-aware metric) is often more interpretable
top_by_pct = aa.top_cell_types_per_subtype(
    summary_st_ct,
    subtype_col=subtype_col,
    celltype_col=celltype_col,
    rank_by='pct_expressing',
    top_n=3,
    min_cells=20,
)
print('=== Top-3 ARC-expressing cell types per subtype (by % expressing) ===')
display(top_by_pct)
top_by_pct.to_csv(cfg.RESULTS_DIR / 'arc_top_celltypes_by_pct.csv', index=False)

## 2b: Luminal vs Basal ARC analysis within epithelial cells

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────#
# Wu's celltype_major labels don't distinguish luminal from basal within
# Cancer/Normal Epithelial. We use the marker scores computed in notebook 05
# (stored in adata.obs) to make this finer distinction.
#
# Biological rationale:
#   Luminal-like cells → more likely ER+/HER2+ origin
#   Basal-like cells   → more likely TNBC/basal-like origin
# ─────────────────────────────────────────────────────────────────────────────

# Check that marker scores exist from notebook 05
score_cols_needed = ['Epithelial_luminal_score', 'Epithelial_basal_score']
missing = [c for c in score_cols_needed if c not in adata.obs.columns]

if missing:
    print(f"WARNING: Missing score columns: {missing}")
    print("These are computed in notebook 05 (score_cell_type_markers).")
    print("Options:")
    print("  1. Re-run notebook 05 first, save h5ad, reload here")
    print("  2. Compute scores now:")
    print("     sc.tl.score_genes(adata, ['EPCAM','KRT8','KRT18','KRT19'], score_name='Epithelial_luminal_score')")
    print("     sc.tl.score_genes(adata, ['KRT5','KRT14','KRT17'], score_name='Epithelial_basal_score')")
else:
    # Subset to epithelial cells only
    epi_mask = adata.obs[celltype_col].isin(['Cancer Epithelial', 'Normal Epithelial'])
    adata_epi = adata[epi_mask].copy()
    print(f"Epithelial cells: {adata_epi.n_obs}")
    print(f"  Cancer Epithelial: {(adata_epi.obs[celltype_col] == 'Cancer Epithelial').sum()}")
    print(f"  Normal Epithelial: {(adata_epi.obs[celltype_col] == 'Normal Epithelial').sum()}")

    # Classify each epithelial cell as luminal or basal dominant
    # Using the difference score: positive = more luminal, negative = more basal
    adata_epi.obs['luminal_basal_diff'] = (
        adata_epi.obs['Epithelial_luminal_score'] -
        adata_epi.obs['Epithelial_basal_score']
    )
    adata_epi.obs['epi_subtype'] = np.where(
        adata_epi.obs['luminal_basal_diff'] > 0,
        'Luminal-like',
        'Basal-like'
    )

    print(f"\nEpithelial subtype classification:")
    print(adata_epi.obs['epi_subtype'].value_counts())
    print(f"\nEpithelial subtype x cancer subtype:")
    print(pd.crosstab(adata_epi.obs['epi_subtype'], adata_epi.obs[subtype_col]))

In [ ]:
# ARC expression in luminal vs basal epithelial cells, split by cancer subtype
if 'epi_subtype' in adata_epi.obs.columns:
    from scipy import sparse

    # Get ARC expression vector
    arc_idx = adata_epi.var_names.get_loc('ARC')
    arc_expr = adata_epi.X[:, arc_idx]
    if sparse.issparse(arc_expr):
        arc_expr = np.asarray(arc_expr.todense()).ravel()
    adata_epi.obs['ARC_expr'] = arc_expr

    # Summary table: subtype x epi_subtype x cancer/normal
    summary_lumbasal = adata_epi.obs.groupby(
        [subtype_col, 'epi_subtype', celltype_col]
    ).agg(
        n_cells=('ARC_expr', 'count'),
        pct_expressing=('ARC_expr', lambda x: round((x > 0).mean() * 100, 2)),
        mean_expr=('ARC_expr', lambda x: round(x.mean(), 4)),
        mean_expr_pos=('ARC_expr', lambda x: round(x[x > 0].mean(), 4) if (x > 0).any() else 0),
    ).reset_index()

    print("=== ARC expression: subtype × luminal/basal × cancer/normal ===")
    display(summary_lumbasal)

    summary_lumbasal.to_csv(
        cfg.RESULTS_DIR / 'arc_luminal_basal_subtype.csv',
        index=False
    )
    print("Saved: results/arc_luminal_basal_subtype.csv")

In [ ]:
# Visualize ARC in luminal vs basal cells
if 'epi_subtype' in adata_epi.obs.columns and 'ARC_expr' in adata_epi.obs.columns:

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))

    # Panel 1: % ARC-positive by epi_subtype and subtype
    pct_data = adata_epi.obs.groupby([subtype_col, 'epi_subtype'])['ARC_expr'].apply(
        lambda x: (x > 0).mean() * 100
    ).reset_index()
    pct_data.columns = [subtype_col, 'epi_subtype', 'pct_arc_positive']

    sns.barplot(
        data=pct_data,
        x=subtype_col,
        y='pct_arc_positive',
        hue='epi_subtype',
        ax=axs[0],
        palette=['#2196F3', '#FF5722'],
    )
    axs[0].set_title('% ARC-positive cells\n(luminal vs basal epithelial)')
    axs[0].set_ylabel('% ARC-positive')
    axs[0].set_xlabel('Cancer subtype')
    axs[0].legend(title='Epithelial type')

    # Panel 2: Mean ARC expression by epi_subtype and subtype
    mean_data = adata_epi.obs.groupby([subtype_col, 'epi_subtype'])['ARC_expr'].mean().reset_index()
    mean_data.columns = [subtype_col, 'epi_subtype', 'mean_ARC']

    sns.barplot(
        data=mean_data,
        x=subtype_col,
        y='mean_ARC',
        hue='epi_subtype',
        ax=axs[1],
        palette=['#2196F3', '#FF5722'],
    )
    axs[1].set_title('Mean ARC expression\n(luminal vs basal epithelial)')
    axs[1].set_ylabel('Mean log-normalized ARC')
    axs[1].set_xlabel('Cancer subtype')
    axs[1].legend(title='Epithelial type')

    # Panel 3: Cancer vs Normal within luminal and basal
    cancer_normal_pct = adata_epi.obs.groupby(
        ['epi_subtype', celltype_col]
    )['ARC_expr'].apply(
        lambda x: (x > 0).mean() * 100
    ).reset_index()
    cancer_normal_pct.columns = ['epi_subtype', celltype_col, 'pct_arc_positive']

    sns.barplot(
        data=cancer_normal_pct,
        x='epi_subtype',
        y='pct_arc_positive',
        hue=celltype_col,
        ax=axs[2],
        palette=['#E91E63', '#4CAF50'],
    )
    axs[2].set_title('% ARC-positive: Cancer vs Normal\n(within luminal/basal)')
    axs[2].set_ylabel('% ARC-positive')
    axs[2].set_xlabel('Epithelial type')
    axs[2].legend(title='Cell type', fontsize=8)

    plt.tight_layout()
    plt.savefig(cfg.FIGURES_DIR / 'arc_luminal_basal_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: figures/arc_luminal_basal_analysis.png")

## 3. Statistically sound analysis: pseudo-bulk by patient

Aggregate counts per (patient, cell type), then compare across subtypes with patient as the unit of replication.

In [ ]:
# Aggregate counts: (patient x cell_type) pseudo-samples
counts_pb, meta_pb = aa.pseudobulk_counts(
    adata,
    group_cols=(cfg.BATCH_KEY, celltype_col),
    counts_layer='counts',
    min_cells=cfg.MIN_CELLS_FOR_PSEUDOBULK,
)
print(f'Pseudo-bulk samples: {len(counts_pb)}')
print(f'Unique patients: {meta_pb[cfg.BATCH_KEY].nunique()}')
print(f'Unique cell types: {meta_pb[celltype_col].nunique()}')

In [ ]:
# Attach subtype labels to pseudo-bulk metadata (one subtype per patient)
patient_to_subtype = (
    adata.obs.groupby(cfg.BATCH_KEY)[subtype_col].first().to_dict()
)
meta_pb[subtype_col] = meta_pb[cfg.BATCH_KEY].map(patient_to_subtype)

# Extract ARC expression (log1p CPM)
arc_pb = aa.pseudobulk_expression(
    counts_pb, meta_pb, gene='ARC', normalize=True
)
print(arc_pb.head())
arc_pb.to_csv(cfg.RESULTS_DIR / 'arc_pseudobulk.csv')

In [ ]:
# Pairwise Wilcoxon tests between subtypes, within each cell type
wilcox = aa.wilcoxon_per_celltype(
    arc_pb,
    gene='ARC',
    subtype_col=subtype_col,
    celltype_col=celltype_col,
    min_samples=3,
)
print('=== Pairwise pseudo-bulk Wilcoxon (ARC across subtypes, per cell type) ===')
display(wilcox)
wilcox.to_csv(cfg.RESULTS_DIR / 'arc_pseudobulk_wilcoxon.csv', index=False)

## 4. Visualizations

In [ ]:
# UMAP colored by ARC expression
sc.pl.umap(
    adata,
    color=['ARC'],
    color_map='magma',
    vmin=0, vmax='p99',  # clip color scale at 99th percentile for visibility
    show=False,
)
plt.savefig(cfg.FIGURES_DIR / 'umap_arc_expression.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dotplot: ARC by cell type, split by subtype
# groupby via combined key gives both groupings in one plot
adata.obs['_subtype_celltype'] = (
    adata.obs[subtype_col].astype(str) + ' | ' + adata.obs[celltype_col].astype(str)
)
sc.pl.dotplot(
    adata,
    var_names=['ARC'],
    groupby='_subtype_celltype',
    standard_scale='var',
    show=False,
)
plt.savefig(cfg.FIGURES_DIR / 'dotplot_arc_subtype_celltype.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Violin: pseudo-bulk ARC expression per subtype, faceted by cell type
g = sns.catplot(
    data=arc_pb,
    x=subtype_col, y='ARC_expr',
    col=celltype_col, col_wrap=4,
    kind='violin', inner='points', height=3, aspect=1,
    sharey=False, cut=0,
)
g.set_axis_labels(subtype_col, 'log1p(CPM) ARC')
g.set_titles('{col_name}')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'violin_arc_pseudobulk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: mean ARC expression (pseudo-bulk) by subtype x cell type
pivot = arc_pb.pivot_table(
    index=celltype_col, columns=subtype_col, values='ARC_expr', aggfunc='mean'
)
plt.figure(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='magma', cbar_kws={'label': 'log1p(CPM) ARC'})
plt.title('Mean pseudo-bulk ARC expression by subtype x cell type')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'heatmap_arc_subtype_celltype.png', dpi=150, bbox_inches='tight')
plt.show()

## Normal Epithelial vs Cancer Epithelial ARC comparison

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

# Filter to epithelial cells only
epi_mask = adata.obs[cfg.CELLTYPE_COLUMN_AUTHORS].isin(
    ['Cancer Epithelial', 'Normal Epithelial']
)
adata_epi = adata[epi_mask].copy()

print(f"Epithelial cells: {adata_epi.n_obs}")
print(adata_epi.obs[cfg.CELLTYPE_COLUMN_AUTHORS].value_counts())
print(adata_epi.obs[cfg.SUBTYPE_COLUMN].value_counts())

In [ ]:
# Build pseudo-bulk for epithelial cells only
# Group by (patient, celltype) — gives one value per patient per cell type
counts_epi, meta_epi = aa.pseudobulk_counts(
    adata_epi,
    group_cols=('orig.ident', cfg.CELLTYPE_COLUMN_AUTHORS),
    counts_layer='counts',
    min_cells=cfg.MIN_CELLS_FOR_PSEUDOBULK,
)

# Add subtype labels
patient_to_subtype = adata.obs.groupby('orig.ident')[cfg.SUBTYPE_COLUMN].first().to_dict()
meta_epi[cfg.SUBTYPE_COLUMN] = meta_epi['orig.ident'].map(patient_to_subtype)

# Extract ARC expression
arc_epi = aa.pseudobulk_expression(counts_epi, meta_epi, gene='ARC', normalize=True)
arc_epi.columns = arc_epi.columns.str.replace(' ', '_')
arc_epi_col = [c for c in arc_epi.columns if 'ARC' in c][0]

print(f"\nPseudo-bulk samples: {len(arc_epi)}")
print(arc_epi[cfg.CELLTYPE_COLUMN_AUTHORS].value_counts())
print(arc_epi.head())

In [ ]:
# ============================================================
# Test 1: Overall — Normal vs Cancer Epithelial (all subtypes)
# ============================================================
cancer_vals = arc_epi.loc[
    arc_epi[cfg.CELLTYPE_COLUMN_AUTHORS] == 'Cancer Epithelial', arc_epi_col
].values

normal_vals = arc_epi.loc[
    arc_epi[cfg.CELLTYPE_COLUMN_AUTHORS] == 'Normal Epithelial', arc_epi_col
].values

stat, p_overall = mannwhitneyu(normal_vals, cancer_vals, alternative='two-sided')

print("=== Overall: Normal vs Cancer Epithelial ===")
print(f"Normal Epithelial:  n={len(normal_vals)}, median={np.median(normal_vals):.3f}")
print(f"Cancer Epithelial:  n={len(cancer_vals)}, median={np.median(cancer_vals):.3f}")
print(f"Median difference:  {np.median(normal_vals) - np.median(cancer_vals):.3f}")
print(f"p-value:            {p_overall:.4f}")
print(f"Interpretation:     {'Significant (p<0.05)' if p_overall < 0.05 else 'Not significant'}")

In [ ]:
# ============================================================
# Test 2: Per subtype — Normal vs Cancer Epithelial
# ============================================================
subtypes = arc_epi[cfg.SUBTYPE_COLUMN].dropna().unique()
results = []

for subtype in sorted(subtypes):
    sub = arc_epi[arc_epi[cfg.SUBTYPE_COLUMN] == subtype]
    
    cancer = sub.loc[
        sub[cfg.CELLTYPE_COLUMN_AUTHORS] == 'Cancer Epithelial', arc_epi_col
    ].values
    normal = sub.loc[
        sub[cfg.CELLTYPE_COLUMN_AUTHORS] == 'Normal Epithelial', arc_epi_col
    ].values
    
    if len(cancer) < 2 or len(normal) < 2:
        print(f"{subtype}: insufficient samples (cancer n={len(cancer)}, normal n={len(normal)})")
        continue
    
    stat, p = mannwhitneyu(normal, cancer, alternative='two-sided')
    
    results.append({
        'subtype': subtype,
        'n_cancer_patients': len(cancer),
        'n_normal_patients': len(normal),
        'median_cancer': np.median(cancer),
        'median_normal': np.median(normal),
        'median_diff (normal - cancer)': np.median(normal) - np.median(cancer),
        'p_value': p,
    })

df_results = pd.DataFrame(results)

# FDR correction across subtypes
if len(df_results) > 0:
    df_results['fdr'] = multipletests(df_results['p_value'], method='fdr_bh')[1]

print("=== Per subtype: Normal vs Cancer Epithelial ===")
print(df_results.round(3).to_string(index=False))

df_results.to_csv(cfg.RESULTS_DIR / 'arc_normal_vs_cancer_epithelial.csv', index=False)
print("\nSaved: results/arc_normal_vs_cancer_epithelial.csv")

In [ ]:
# ============================================================
# Visualization: violin plot Normal vs Cancer per subtype
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Overall
sns.violinplot(
    data=arc_epi,
    x=cfg.CELLTYPE_COLUMN_AUTHORS,
    y=arc_epi_col,
    inner='quartile', cut=0,
    palette={'Cancer Epithelial': '#2ca02c', 'Normal Epithelial': '#8c564b'},
    ax=axs[0],
)
axs[0].set_title(f'Overall\np={p_overall:.4f}', fontsize=11)
axs[0].set_xlabel('')
axs[0].set_ylabel('ARC log1p(CPM) — pseudo-bulk')
axs[0].tick_params(axis='x', rotation=20)

# Panel 2: Per subtype — cleaner grouping
sns.violinplot(
    data=arc_epi,
    x=cfg.SUBTYPE_COLUMN,
    y=arc_epi_col,
    hue=cfg.CELLTYPE_COLUMN_AUTHORS,
    inner='quartile', cut=0,
    palette={'Cancer Epithelial': '#2ca02c', 'Normal Epithelial': '#8c564b'},
    split=False,
    ax=axs[1],
)
axs[1].set_title('Per subtype', fontsize=11)
axs[1].set_xlabel('')
axs[1].set_ylabel('ARC log1p(CPM) — pseudo-bulk')
axs[1].legend(title='', loc='upper right', fontsize=8)

# Add p-values above each subtype
for i, row in df_results.iterrows():
    axs[1].text(
        i, axs[1].get_ylim()[1] * 0.97,
        f"p={row['p_value']:.3f}\n(n={int(row['n_cancer_patients'])}C/{int(row['n_normal_patients'])}N)",
        ha='center', fontsize=7.5,
    )

plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'arc_normal_vs_cancer_epithelial_clean.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary of findings

### ARC expression by subtype (per-cell statistics)
ARC expression was highest in ER+ tumors and lowest in TNBC across most cell types. Within epithelial populations, Normal Epithelial cells showed consistently higher ARC expression (19-25%) than Cancer Epithelial cells (3-8%), suggesting ARC downregulation may be associated with malignant transformation."
- Highest mean ARC expression (subtype): *ER*
- Lowest mean ARC expression (subtype): *TNBC*
- % ARC-expressing cells (by subtype, across all cell types):
  - ER+: *~6.8%*
  - HER2+: *~6.6%*
  - TNBC: *~2.6%*
- Normal Epithelial cells have the highest ARC expression across all subtypes:
  - ER+: *19.2%*
  - HER2+: *24.8%*
  - TNBC: *18%*
- Cancer Epithelial show subtypes differences in ARC:
  - ER+: *8%*
  - HER2+: *3.5%*
  - TNBC: *2.9%*

### ARC expression by cell type (pooled across subtypes)
- Highest ARC: Normal Epithelial, PVL, Endothelial, CAFs
- Lowest ARC: B cells, Plasmablasts, T cells (<2% across all subtypes)

### Pseudo-bulk statistical analysis
- No pairwise comparison survived FDR correction (all FDR > 0.25)
- Closest to significance: T cells HER2+ vs TNBC (p=0.011, FDR=0.26,
  median difference=1.70 log1p(CPM))
- Consistent directional trend across 5+ cell types: TNBC < ER+ ≈ HER2+
- Statistical non-significance reflects low power (3-11 patients per
  subtype per cell type); findings are exploratory

### Consistency with published literature
- The ER+ > TNBC direction is consistent with Yee et al. 2025
  (World J Oncol, PMC11750752), who found significantly higher ARC
  in ER+/HER2- vs other subtypes in TCGA (n=1069) and SCAN-B (n=3273)
- Yee et al. also reported Normal Epithelial as the main ARC-expressing
  cell type in GSE176078 (SCP1039) — the same dataset analyzed here

### Caveats
- DEV_MODE subsampling (~36K of 100K cells) used for local execution
- ARC is sparsely expressed (median = 0 in all groups); pct_expressing
  is the primary metric
- Pseudo-bulk tests have low power with n=3-11 patients per subtype
- Results are exploratory; replication in larger cohorts required

### Normal vs Cancer Epithelial ARC comparison

**Overall** (all subtypes pooled):
- Normal Epithelial median: 3.024 log1p(CPM), n=10 patients
- Cancer Epithelial median: 1.341 log1p(CPM), n=20 patients
- Median difference: +1.683 (Normal higher)
- p=0.0615 — strong trend, does not reach p<0.05 threshold
- Limited power: only 10 patients contributed Normal Epithelial
  pseudo-bulk samples (many patients lack detectable normal cells)

**Per subtype**:
- ER+: minimal difference (Normal 2.504 vs Cancer 2.380, diff=+0.124, p=0.825)
  → ARC largely preserved in ER+ cancer cells
- TNBC: moderate difference (Normal 2.211 vs Cancer 0.760, diff=+1.451, p=0.368)
  → ARC substantially lower in TNBC cancer vs normal epithelial
- HER2+: large apparent difference (diff=+3.243) but n=2 normal vs n=3 cancer
  → insufficient power to interpret

**Biological interpretation**:
ARC expression in Cancer Epithelial cells is lower than in Normal
Epithelial cells within the same tumor microenvironment (trend p=0.06),
with the largest loss observed in TNBC — the most de-differentiated
and aggressive subtype. ER+ cancer cells retain ARC expression at
levels closer to normal epithelium, consistent with their more
differentiated phenotype. This pattern is concordant with Yee et al.
2025 who found ARC highest in ER+/HER2- and lowest in TNBC across
two independent bulk RNA cohorts.

**Caveat**: The overall test falls just short of significance (p=0.06),
reflecting small normal epithelial sample size (n=10). A larger
cohort would be required to confirm this finding.